# Week 103, Day 4: Graduation project showcase

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w103d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('showcase.py', 'w', encoding='utf-8') as f:
    f.write('"""A graduation showcase: who presents what, and the feedback afterwards."""\n\nTALKS = [\n    {"presenter": "Amara", "project": "Waymark", "minutes": 10},\n    {"presenter": "Bo", "project": "helpdesk", "minutes": 10},\n    {"presenter": "Chidi", "project": "spamfilter", "minutes": 8},\n    {"presenter": "Dana", "project": "bikehire", "minutes": 10},\n    {"presenter": "Eli", "project": "spend", "minutes": 6},\n    {"presenter": "Femi", "project": "tidepool", "minutes": 10},\n]\n\n# Each listener rates each talk from 1 to 5 for clarity, the demo and depth.\nFEEDBACK = [\n    {"talk": "Amara", "clarity": 5, "demo": 4, "depth": 5, "comment": "Loved the round walks demo"},\n    {"talk": "Amara", "clarity": 4, "demo": 5, "depth": 4, "comment": "Clear diagrams"},\n    {"talk": "Bo", "clarity": 3, "demo": 2, "depth": 4, "comment": "Demo login failed, backup video saved it"},\n    {"talk": "Bo", "clarity": 4, "demo": 3, "depth": 5, "comment": "Great on security"},\n    {"talk": "Chidi", "clarity": 5, "demo": 5, "depth": 3, "comment": "Would like more on the maths"},\n    {"talk": "Dana", "clarity": 2, "demo": 4, "depth": 4, "comment": "Too fast"},\n    {"talk": "Dana", "clarity": 3, "demo": 4, "depth": 3, "comment": "Slides too dense"},\n    {"talk": "Eli", "clarity": 5, "demo": 5, "depth": 2, "comment": "Short and sweet"},\n]\n')

**Predict before you run:** what will this print?



In [ ]:
from showcase import TALKS, FEEDBACK

print(sum(t["minutes"] for t in TALKS), len({f["talk"] for f in FEEDBACK}), [t["presenter"] for t in TALKS if t["minutes"] < 10])

> **Key idea.** Run a showcase to time: slots from a start time, a setup gap between talks and a break every few. Shape a ten-minute talk as problem, what it does, demo, how it works and what you learned. Script and check the demo, and keep a backup for when it fails. Take questions so everyone asks once before anyone asks twice; repeat, answer briefly, and say when you don't know. Average feedback per criterion and work on the lowest.

### Exercise: The programme

Write `programme(talks, start="14:00", gap=2, every=4, rest=10)`: a list of `(time, presenter, project)` for each talk in order, `time` as `HH:MM`. Each talk starts when the previous one ends plus `gap` minutes; after every `every` talks (but not after the last) there's a break of `rest` minutes instead of the gap.

In [ ]:
from showcase import TALKS

def programme(talks, start="14:00", gap=2, every=4, rest=10):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from showcase import TALKS\n\ndef test_programme():\n    \"slots, gaps and a break\"\n    slots = programme(TALKS)\n    assert slots == [(\"14:00\", \"Amara\", \"Waymark\"), (\"14:12\", \"Bo\", \"helpdesk\"), (\"14:24\", \"Chidi\", \"spamfilter\"),\n                     (\"14:34\", \"Dana\", \"bikehire\"), (\"14:54\", \"Eli\", \"spend\"), (\"15:02\", \"Femi\", \"tidepool\")]\n    assert programme(TALKS[:2], start=\"09:55\", gap=0) == [(\"09:55\", \"Amara\", \"Waymark\"), (\"10:05\", \"Bo\", \"helpdesk\")]\n    assert programme(TALKS[:4], every=4)[-1][0] == \"14:34\" and programme([]) == []")

### Exercise: Rehearsing to time

A rehearsal records the minute at which each section **ended** (`marks`, a list of numbers from the start). Write `over_time(plan, marks)` where `plan` is a list of `(section, minutes)`: the sections that took more than 20% longer than planned, with how many minutes over (1 place), as `(section, over)`. Raise `ValueError` if the lengths differ.

In [ ]:
def over_time(plan, marks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PLAN = [(\"problem\", 1), (\"what it does\", 2), (\"demo\", 4), (\"how it works\", 2), (\"what I learned\", 1)]\n\ndef test_pace():\n    \"each section's own length\"\n    assert over_time(PLAN, [1.1, 3.0, 8.5, 11.0, 11.8]) == [(\"demo\", 1.5), (\"how it works\", 0.5)]\n    assert over_time(PLAN, [1, 3, 7, 9, 10]) == []\n    assert over_time(PLAN, [1.3, 3.3, 7.3, 9.3, 10.3]) == [(\"problem\", 0.3)]\n    try:\n        over_time(PLAN, [1, 2])\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"lengths\")")

### Exercise: Checking the demo script

A demo script is a list of `(command, expected)`; `run_step(command)` returns what the command shows (or raises an exception). Write `check_demo(script, run_step)`: `("ok", None)` if every step shows what it should, otherwise `("use backup", n)` for the first failing step `n` (from 1), where a step fails if it shows something else **or** raises. Stop at the first failure: don't run later steps.

In [ ]:
def check_demo(script, run_step):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SCRIPT = [(\"waymark route Mill Ridge\", \"6.0 km\"), (\"waymark time Mill Ridge\", \"1 h 52 min\"), (\"waymark share 7\", \"code ready\")]\n\ndef test_demo():\n    \"first failure, then stop\"\n    calls = []\n    def good(command):\n        calls.append(command)\n        return dict(SCRIPT)[command]\n    assert check_demo(SCRIPT, good) == (\"ok\", None) and len(calls) == 3\n    calls.clear()\n    def wrong(command):\n        calls.append(command)\n        return \"6.5 km\" if \"route\" in command else \"?\"\n    assert check_demo(SCRIPT, wrong) == (\"use backup\", 1) and len(calls) == 1\n    def offline(command):\n        if \"share\" in command:\n            raise ConnectionError(\"no network\")\n        return dict(SCRIPT)[command]\n    assert check_demo(SCRIPT, offline) == (\"use backup\", 3) and check_demo([], good) == (\"ok\", None)")

### Exercise: Questions, fairly

`hands` is the list of names in the order hands went up; someone can raise their hand more than once. Write `question_order(hands, limit=None)`: the order to take them so everyone asks once before anyone asks twice (and so on), keeping the order hands went up within each round. With `limit`, return only the first `limit`.

In [ ]:
def question_order(hands, limit=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"rounds\"\n    hands = [\"Kofi\", \"Lin\", \"Kofi\", \"Kofi\", \"Mo\", \"Lin\", \"Nia\"]\n    assert question_order(hands) == [\"Kofi\", \"Lin\", \"Mo\", \"Nia\", \"Kofi\", \"Lin\", \"Kofi\"]\n    assert question_order(hands, limit=4) == [\"Kofi\", \"Lin\", \"Mo\", \"Nia\"] and question_order([]) == []\n    assert question_order([\"Mo\", \"Mo\", \"Lin\"]) == [\"Mo\", \"Lin\", \"Mo\"]")

### Exercise: What the feedback says

Write `summary(feedback, talks)`: a dict from each presenter in `talks` (in order) to a dict with `clarity`, `demo` and `depth` (averages to 1 place), `work_on` (the lowest criterion; ties go to the first in that order) and `n` (how many forms). A talk with no forms gets `None` for each average and `work_on`, and `n` of 0.

In [ ]:
from showcase import TALKS, FEEDBACK

CRITERIA = ["clarity", "demo", "depth"]

def summary(feedback, talks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from showcase import TALKS, FEEDBACK\n\ndef test_summary():\n    \"averages and the lowest\"\n    s = summary(FEEDBACK, TALKS)\n    assert list(s) == [\"Amara\", \"Bo\", \"Chidi\", \"Dana\", \"Eli\", \"Femi\"]\n    assert s[\"Amara\"] == {\"clarity\": 4.5, \"demo\": 4.5, \"depth\": 4.5, \"work_on\": \"clarity\", \"n\": 2}\n    assert s[\"Bo\"][\"work_on\"] == \"demo\" and s[\"Bo\"][\"demo\"] == 2.5 and s[\"Dana\"][\"work_on\"] == \"clarity\"\n    assert s[\"Eli\"][\"work_on\"] == \"depth\" and s[\"Chidi\"][\"n\"] == 1\n    assert s[\"Femi\"] == {\"clarity\": None, \"demo\": None, \"depth\": None, \"work_on\": None, \"n\": 0}")

### Bug hunt: Nine past two

The start time prints as `14:9`. Pad the minutes so it prints `14:09`.

In [ ]:
now = 14 * 60 + 9
print(f"{now // 60}:{now % 60}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"two digits\"\n    expect_output(\"14:09\")")

### Bug hunt: A pattern from one comment

Comments are counted for words that come up again, but the count is per word and doesn't ignore case, so `Too fast` and `too dense` miss each other and this prints `[]`. Lower-case the words so it prints `['too']`.

In [ ]:
from collections import Counter

comments = ["Too fast", "Slides too dense", "Clear diagrams"]
words = Counter(word for c in comments for word in c.split())
print([w for w, n in words.items() if n >= 2])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"case\"\n    expect_output(\"['too']\")")

### Puzzle: Ten minutes

Arrange the sections of a ten-minute project talk. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # minutes of demo in the ten-minute shape above (an int)
q = ___   # what to do first when someone asks a question (a short phrase)
r = ___   # True or False: if a live demo fails, start the talk again
s = ___   # how many presenters in TALKS have no feedback (an int)
t = ___   # the criterion Bo should work on

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4 and \"repeat\" in q.lower() and r is False, \"p, q and r\"\n    assert s == 1 and t == \"demo\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the showcase report

Write `report(talks, feedback, start="14:00")`: the text of a report, one block per talk in programme order, with blocks separated by a blank line. Each block is `TIME PRESENTER: PROJECT` then, if there's feedback, `  clarity C, demo D, depth P (N forms)` and `  work on: CRITERION`, or, if there isn't, `  no feedback yet`. After the blocks, a blank line and `Comments mentioning "too": N` (the number of comments containing the word `too`, in any case). Use `programme` (with its defaults) and `summary` from today.

In [ ]:
from showcase import TALKS, FEEDBACK

CRITERIA = ["clarity", "demo", "depth"]

def report(talks, feedback, start="14:00"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from showcase import TALKS, FEEDBACK\n\ndef test_report():\n    \"the whole report\"\n    text = report(TALKS, FEEDBACK)\n    blocks = text.split(\"\\n\\n\")\n    assert len(blocks) == 7 and blocks[-1] == 'Comments mentioning \"too\": 2'\n    assert blocks[0] == \"14:00 Amara: Waymark\\n  clarity 4.5, demo 4.5, depth 4.5 (2 forms)\\n  work on: clarity\"\n    assert blocks[2] == \"14:24 Chidi: spamfilter\\n  clarity 5.0, demo 5.0, depth 3.0 (1 form)\\n  work on: depth\"\n    assert blocks[5] == \"15:02 Femi: tidepool\\n  no feedback yet\"\n    assert report(TALKS[:1], FEEDBACK, start=\"09:30\").startswith(\"09:30 Amara\")")

## Warm-up

Fifty-four minutes of talks, feedback for five of the six, and two talks shorter than the usual ten. Femi has no feedback yet: a programme has to cope with gaps like that without crashing or quietly leaving the talk out.

## The programme

A showcase runs on time or not at all: the last speaker shouldn't lose their slot because the first one ran over. So the programme lists each talk's start time, with a **gap** between talks for the next speaker to set up (two minutes is enough) and a **break** after every few talks, since listeners tire. The times are worked out from the start time in minutes, then formatted as `HH:MM` with zero padding.

For your own talk: ten minutes is about 1,200 spoken words, or one idea per minute. A common shape is **problem** (1 minute), **what it does** (2), **demo** (4), **how it works** (2) and **what you learned** (1). Rehearse with a timer and note when each section ends, then cut whatever runs long; the demo is usually the culprit.

## A demo that can fail safely

Live demos fail: the network drops, a login has expired, the projector changes the screen size. Plan for it:

- **Script the demo**: the exact commands or clicks, and what each should show.
- **Check the script** just before you go on, step by step, and stop at the first step that doesn't show what it should.
- **Keep a backup**: a recording or screenshots of each step. If a step fails live, say so calmly ("the login has expired; here's what it does") and switch to the backup. Listeners remember a calm recovery, not the failure.
- **Use sample data** you've prepared, never real personal data.

## Questions

Take questions **fairly**: if several people want to ask, let everyone ask once before anyone asks twice, keeping the order they raised their hands. Then answer:

- **Repeat the question** so everyone hears it, and so you're sure you've understood it.
- **Answer briefly**, then check: "does that answer it?"
- **Say "I don't know"** when you don't, and offer to find out. An honest "I haven't measured that" is better than a guess.
- **Take a hostile or off-topic question** politely, and offer to talk afterwards.

## Feedback

A simple form works: ratings from 1 to 5 for **clarity**, **the demo** and **depth**, and a comment. Average each rating per talk (to 1 decimal place), find each talk's **lowest** criterion (that's the one to work on), and read the comments for patterns. One "too fast" is a data point; two is a finding.

## Practice

## Quiz

## Daily challenge

## Your showcase today

1. **Choose the project** you'll present: usually the strongest one from yesterday's portfolio.
2. **Write the ten-minute shape** and rehearse it twice with a timer, noting when each section ends.
3. **Script the demo**, check it, and record a backup.
4. **Present it** to someone: a friend, a study group, a local meet-up, or a recording you share. Ask for the three ratings and a comment, and write down what you'll change.

## Before you go

Tomorrow looks ahead: where to go after the course, how to keep learning, and how to plan your next year.